[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frbennett/shapleyx/blob/main/workshops/transform_sensitivity.ipynb)

# NSE Sensitivity: Square-Root Transformed vs Raw Discharge
## SAC-SMA Model — St Helens Creek, North Queensland

**Workshop notebook.** Click the Colab badge above to run in your browser.

**Question:** Does the Box-Cox transform ($\lambda = 0.5$, square-root) change which parameters appear most sensitive? The sqrt-transform stabilises error variance across four orders of magnitude of discharge. Without it, raw NSE is dominated by the largest flows.

This notebook compares Shapley-effect sensitivity rankings for:
- **Sqrt-NSE:** NSE on $\sqrt{Q}$ (variance-stabilised)
- **Raw-NSE:** NSE on $Q$ directly (high-flow dominated)

---

### Pipeline

| Phase | Description |
|-------|-------------|
| **1. Design** | QMC Sobol' (4,096 pts in 14-D) |
| **2. Evaluate** | Run SAC-SMA → compute both NSE variants |
| **3. Surrogate** | RS-HDMR (OMP-CV) for each variant |
| **4. Sensitivity** | MC Shapley effects (Vrugt correlations) |
| **5. Compare** | Side-by-side rankings, parameter shifts |

In [ ]:
# ── One-time setup (Colab / remote only) ──
import sys, os, urllib.request

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print('Installing shapleyx...')
    !pip install -q shapleyx
    REPO = 'https://raw.githubusercontent.com/frbennett/shapleyx/main/workshops/transform_data'
    os.makedirs('data', exist_ok=True)
    for f in ['sacramento.py', 'st_helens_forcing_original.csv', 'es_parameters.csv']:
        urllib.request.urlretrieve(f'{REPO}/{f}', f'data/{f}')
    print('Data downloaded.')
else:
    print('Running locally.')

# ── Imports ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats.qmc import Sobol
import warnings, time
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight'})

from shapleyx import rshdmr
from shapleyx.utilities.mc_shapley import GaussianCopulaUniform, shapley_effects

sys.path.insert(0, 'data')
from sacramento import sacsma

print('All imports OK')

---
## 1. Load Forcing Data

In [ ]:
df = pd.read_csv('data/st_helens_forcing_original.csv', parse_dates=['date'], dayfirst=True)
prcp = df['rainfall'].values.astype(float)
pet  = df['pet'].values.astype(float)
Q_obs = df['Q_CUMEC'].values.astype(float)

AREA = 120500000.0
WARMUP = 365

Q_eval = Q_obs[WARMUP:]          # post-warmup observed
Y_sqrt_obs = Q_eval ** 0.5        # sqrt-transform target

print(f'Forcing: {len(df)} timesteps')
print(f'Evaluation: {len(Q_eval)} days (post-warmup)')
print(f'Q range: [{Q_eval.min():.3f}, {Q_eval.max():.1f}] m³/s')

---
## 2. Parameter Definitions

In [ ]:
param_names = ['UZTWM','UZFWM','LZTWM','LZFPM','LZFSM','ADIMP',
               'UZK','LZPK','LZSK','ZPERC','REXP','PCTIM','PFREE','SIDE']
D = len(param_names)

lo = np.array([30, 10, 20, 40, 15, 0, 0.2, 0.001, 0.03, 0, 0, 0, 0, 0])
hi = np.array([125, 75, 300, 600, 300, 0.2, 0.9, 0.03, 0.8, 80, 5, 0.05, 0.8, 0.8])

print(f'{D} parameters')

---
## 3. NSE Functions

Both evaluate on all post-warmup days. The only difference is the transform.

In [ ]:
def mm_to_cumec(flow_mm):
    return flow_mm / 1000 * AREA / (3600 * 24)


def nse_sqrt(params):
    """NSE on sqrt-transformed discharge."""
    sim = mm_to_cumec(sacsma(prcp, pet, np.asarray(params, float))[WARMUP:])
    y_sim = sim ** 0.5
    ss_res = np.sum((Y_sqrt_obs - y_sim)**2)
    ss_tot = np.sum((Y_sqrt_obs - Y_sqrt_obs.mean())**2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else -np.inf


def nse_raw(params):
    """NSE on raw (non-transformed) discharge."""
    sim = mm_to_cumec(sacsma(prcp, pet, np.asarray(params, float))[WARMUP:])
    ss_res = np.sum((Q_eval - sim)**2)
    ss_tot = np.sum((Q_eval - Q_eval.mean())**2)
    return 1 - ss_res / ss_tot if ss_tot > 0 else -np.inf


# Warm-up (compiles numba)
p0 = np.array([44.13,52.76,80.87,112.70,52.81,0.084,0.490,0.0148,0.133,35.31,2.532,0.00578,0.424,0.0093])
print(f'NSE sqrt: {nse_sqrt(p0):.4f}  |  NSE raw: {nse_raw(p0):.4f}')

---
## 4. Training Design — QMC Sobol'

In [ ]:
N = 4096
X_u = Sobol(d=D, scramble=True, seed=42).random(n=N)
X = lo + X_u * (hi - lo)
print(f'{N} Sobol points in {D}-D parameter space')

---
## 5. Evaluate SAC-SMA

~40s on 8 cores. Each run produces both NSE values.

In [ ]:
print(f'Evaluating {N} parameter sets...')
t0 = time.time()

Y_sqrt = np.empty(N)
Y_raw  = np.empty(N)

for i in range(N):
    Y_sqrt[i] = nse_sqrt(X[i])
    Y_raw[i]  = nse_raw(X[i])
    if (i+1) % 1000 == 0:
        print(f'  [{i+1:4d}/{N}] {time.time()-t0:.0f}s')

print(f'\nDone in {time.time()-t0:.0f}s')
print(f'\nY_sqrt: mean={Y_sqrt.mean():.4f}, std={Y_sqrt.std():.4f}')
print(f'Y_raw:  mean={Y_raw.mean():.4f}, std={Y_raw.std():.4f}')
print(f'Variance ratio (raw/sqrt): {Y_raw.var()/Y_sqrt.var():.1f}x')

In [ ]:
# Distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(Y_sqrt, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(Y_sqrt.mean(), color='firebrick', ls='--', lw=1.5, label=f'μ={Y_sqrt.mean():.3f}')
ax1.set_xlabel('NSE'); ax1.set_title(f'Sqrt-NSE (σ²={Y_sqrt.var():.4f})'); ax1.legend(fontsize=9)
ax2.hist(Y_raw, bins=40, color='darkorange', edgecolor='white', alpha=0.8)
ax2.axvline(Y_raw.mean(), color='firebrick', ls='--', lw=1.5, label=f'μ={Y_raw.mean():.3f}')
ax2.set_xlabel('NSE'); ax2.set_title(f'Raw NSE (σ²={Y_raw.var():.4f})'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## 6. RS-HDMR Surrogate — Sqrt-NSE

In [ ]:
df_s = pd.DataFrame(X, columns=param_names)
df_s['Y'] = Y_sqrt

t0 = time.time()
surr_sqrt = rshdmr(df_s, polys=[10,6,2], method='omp_cv', n_iter=400, n_jobs=4, verbose=True)
a, b, c =surr_sqrt.run_all()
print(f'\nFitted in {time.time()-t0:.0f}s')

In [ ]:
yp_s = surr_sqrt.predict(X)
r2_s = 1 - np.sum((Y_sqrt - yp_s)**2) / np.sum((Y_sqrt - Y_sqrt.mean())**2)
print(f'Sqrt surrogate: R²={r2_s:.4f}, RMSE={np.sqrt(np.mean((Y_sqrt-yp_s)**2)):.6f}')

---
## 7. RS-HDMR Surrogate — Raw NSE

In [ ]:
df_r = pd.DataFrame(X, columns=param_names)
df_r['Y'] = Y_raw

t0 = time.time()
surr_raw = rshdmr(df_r, polys=[10,6,2], method='omp_cv', n_iter=400, n_jobs=4, verbose=True)
d, e, f =surr_raw.run_all()
print(f'\nFitted in {time.time()-t0:.0f}s')

In [ ]:
yp_r = surr_raw.predict(X)
r2_r = 1 - np.sum((Y_raw - yp_r)**2) / np.sum((Y_raw - Y_raw.mean())**2)
print(f'Raw surrogate:  R²={r2_r:.4f}, RMSE={np.sqrt(np.mean((Y_raw-yp_r)**2)):.6f}')

---
## 8. Surrogate Summary

In [ ]:
print(f'{"":>20s}  {"Sqrt-NSE":>12s}  {"Raw-NSE":>12s}')
print(f'{"":>20s}  {"─"*12}  {"─"*12}')
print(f'{"R²":>20s}  {r2_s:>12.4f}  {r2_r:>12.4f}')
print(f'{"NSE mean":>20s}  {Y_sqrt.mean():>12.4f}  {Y_raw.mean():>12.4f}')
print(f'{"NSE variance":>20s}  {Y_sqrt.var():>12.6f}  {Y_raw.var():>12.6f}')

---
## 9. Vrugt et al. (2006) Correlation Matrix

In [ ]:
vrugt_order = ['UZTWM','UZFWM','UZK','PCTIM','ADIMP','ZPERC','REXP',
               'LZTWM','LZFSM','LZFPM','LZSK','LZPK','PFREE']

C13 = np.array([
    [ 1.00,-0.02,-0.04, 0.19, 0.36,-0.01, 0.11,-0.84,-0.17, 0.28, 0.17, 0.16, 0.57],
    [-0.02, 1.00,-0.53, 0.05, 0.01, 0.15, 0.11,-0.01,-0.08, 0.22,-0.15,-0.18,-0.33],
    [-0.04,-0.53, 1.00,-0.01,-0.41,-0.13,-0.03, 0.07,-0.11,-0.01,-0.02, 0.08, 0.08],
    [ 0.19, 0.05,-0.01, 1.00,-0.16, 0.01, 0.09,-0.14, 0.01, 0.02,-0.08, 0.02,-0.12],
    [ 0.36, 0.01,-0.41,-0.16, 1.00, 0.07, 0.04,-0.21,-0.15, 0.09, 0.17, 0.15, 0.31],
    [-0.01, 0.15,-0.13, 0.01, 0.07, 1.00, 0.05, 0.06,-0.06,-0.10,-0.13, 0.02, 0.00],
    [ 0.11, 0.11,-0.03, 0.09, 0.04, 0.05, 1.00,-0.03, 0.65, 0.68, 0.24,-0.10, 0.24],
    [-0.84,-0.01, 0.07,-0.14,-0.21, 0.06,-0.03, 1.00, 0.28,-0.16,-0.14,-0.23,-0.63],
    [-0.17,-0.08,-0.11, 0.01,-0.15,-0.06, 0.65, 0.28, 1.00, 0.41,-0.33,-0.55,-0.36],
    [ 0.28, 0.22,-0.01, 0.02, 0.09,-0.10, 0.68,-0.16, 0.41, 1.00, 0.06,-0.43,-0.26],
    [ 0.17,-0.15,-0.02,-0.08, 0.17,-0.13, 0.24,-0.14,-0.33, 0.06, 1.00, 0.60, 0.41],
    [ 0.16,-0.18, 0.08, 0.02, 0.15, 0.02,-0.10,-0.23,-0.55,-0.43, 0.60, 1.00, 0.56],
    [ 0.57,-0.33, 0.08,-0.12, 0.31, 0.00, 0.24,-0.63,-0.36,-0.26, 0.41, 0.56, 1.00],
])

vi = {n: vrugt_order.index(n) for n in vrugt_order}
corr = np.eye(D)
for i, ni in enumerate(param_names):
    for j, nj in enumerate(param_names):
        if ni in vi and nj in vi:
            corr[i,j] = C13[vi[ni], vi[nj]]

# Ensure positive definiteness
ev, evec = np.linalg.eigh(corr)
ev = np.maximum(ev, 1e-8)
corr = evec @ np.diag(ev) @ evec.T
corr = corr / np.outer(np.sqrt(np.diag(corr)), np.sqrt(np.diag(corr)))
print(f'Correlation matrix valid (min λ = {np.linalg.eigvalsh(corr).min():.2e})')

---
## 10. MC Shapley Effects — Sqrt-NSE

In [ ]:
joint = GaussianCopulaUniform(lows=lo, highs=hi, corr=corr)

print('Computing MC Shapley — Sqrt-NSE...')
t0 = time.time()

eff_s, sh_s, var_s, lo_s, hi_s = shapley_effects(
    f=surr_sqrt.predict, joint=joint,
    N=2000, method='exhaustive', B=500, predict_batch=surr_sqrt.predict,
    k_max=3, alpha=0.05, random_state=42, progress=True)

print(f'Done in {time.time()-t0:.0f}s | Σ Shapley = {sh_s.sum():.6f}')

df_s_out = pd.DataFrame({'Param': param_names, 'Shapley': eff_s, 'CI_lo': lo_s, 'CI_hi': hi_s})
df_s_out = df_s_out.sort_values('Shapley', ascending=False)
display(df_s_out.round(5))

---
## 11. MC Shapley Effects — Raw NSE

In [ ]:
print('Computing MC Shapley — Raw NSE...')
t0 = time.time()

eff_r, sh_r, var_r, lo_r, hi_r = shapley_effects(
    f=surr_raw.predict, joint=joint,
    N=2000, method='exhaustive', B=500, predict_batch=surr_raw.predict,
    k_max=3, alpha=0.05, random_state=42, progress=True)

print(f'Done in {time.time()-t0:.0f}s | Σ Shapley = {sh_r.sum():.6f}')

df_r_out = pd.DataFrame({'Param': param_names, 'Shapley': eff_r, 'CI_lo': lo_r, 'CI_hi': hi_r})
df_r_out = df_r_out.sort_values('Shapley', ascending=False)
display(df_r_out.round(5))

---
## 12. Side-by-Side Comparison

In [ ]:
comp = pd.DataFrame({'Param': param_names, 'Sqrt': eff_s, 'Raw': eff_r})
comp['Rk_sqrt'] = comp['Sqrt'].rank(ascending=False).astype(int)
comp['Rk_raw']  = comp['Raw'].rank(ascending=False).astype(int)
comp['ΔRank']   = comp['Rk_sqrt'] - comp['Rk_raw']
comp['ΔShapley'] = comp['Raw'] - comp['Sqrt']
comp = comp.sort_values('Sqrt', ascending=False)

print(f'{"Param":>8s}  {"Sqrt":>10s}  {"Rk":>3s}  {"Raw":>10s}  {"Rk":>3s}  {"ΔRk":>4s}  {"ΔShapley":>10s}')
print('─'*60)
for _, r in comp.iterrows():
    print(f'{r["Param"]:>8s}  {r["Sqrt"]:10.6f}  {r["Rk_sqrt"]:3d}  '
          f'{r["Raw"]:10.6f}  {r["Rk_raw"]:3d}  {r["ΔRank"]:+4d}  {r["ΔShapley"]:+10.6f}')

print(f'\nCorrelation(Sqrt, Raw Shapley): {np.corrcoef(sh_s, sh_r)[0,1]:.3f}')

In [ ]:
# Side-by-side bars
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
df_l = comp.sort_values('Sqrt')
ax1.barh(range(D), df_l['Sqrt'], color=plt.cm.Blues(np.linspace(0.3,0.9,D)), height=0.7)
ax1.set_yticks(range(D)); ax1.set_yticklabels(df_l['Param']); ax1.set_title('Sqrt-NSE'); ax1.grid(alpha=0.3,axis='x')
df_r2 = comp.set_index('Param').loc[df_l['Param']].reset_index()
ax2.barh(range(D), df_r2['Raw'], color=plt.cm.Oranges(np.linspace(0.3,0.9,D)), height=0.7)
ax2.set_yticks(range(D)); ax2.set_yticklabels(df_r2['Param']); ax2.set_title('Raw NSE'); ax2.grid(alpha=0.3,axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# Rank shift plot
fig, ax = plt.subplots(figsize=(8, 6))
df_shift = comp.sort_values('ΔRank', key=abs, ascending=False)
colors = ['firebrick' if s > 0 else 'steelblue' for s in df_shift['ΔRank']]
ax.barh(range(D), df_shift['ΔRank'], color=colors, height=0.7)
ax.set_yticks(range(D)); ax.set_yticklabels(df_shift['Param'])
ax.set_xlabel('Rank Shift (Sqrt − Raw)'); ax.set_title('Parameter Importance Shift: Sqrt → Raw NSE')
ax.axvline(0, color='black', lw=0.8)
ax.text(0.98,0.95,'← Raw gains', transform=ax.transAxes, ha='right', fontsize=9, color='firebrick', style='italic')
ax.text(0.02,0.95,'Sqrt gains →', transform=ax.transAxes, ha='left', fontsize=9, color='steelblue', style='italic')
for i, (v, rs, rr) in enumerate(zip(df_shift['ΔRank'], df_shift['Rk_sqrt'], df_shift['Rk_raw'])):
    ax.text(v+(0.5 if v>=0 else -0.5), i, f'#{rs}→#{rr}', va='center', fontsize=7, color='#555')
ax.grid(alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

In [ ]:
# Scatter
fig, ax = plt.subplots(figsize=(7, 7))
mx = max(comp['Sqrt'].max(), comp['Raw'].max()) * 1.1
ax.plot([0,mx],[0,mx],'k--',lw=1,alpha=0.4,label='1:1')
for _, r in comp.iterrows():
    s = abs(r['ΔRank'])
    c = 'firebrick' if s>=3 else ('darkorange' if s>=2 else 'steelblue')
    ax.scatter(r['Sqrt'],r['Raw'],c=c,s=80+s*15,edgecolors='white',lw=0.5,zorder=3)
    ax.annotate(r['Param'],(r['Sqrt'],r['Raw']),textcoords='offset points',xytext=(5,5),fontsize=7,alpha=0.8)
ax.set_xlabel('Sqrt-NSE Shapley'); ax.set_ylabel('Raw-NSE Shapley')
ax.set_title('Sqrt-Transformed vs Raw NSE'); ax.legend(fontsize=8)
ax.set_xlim(0,mx); ax.set_ylim(0,mx); ax.set_aspect('equal'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 13. Top Shifters & Interpretation

Parameters above the 1:1 line gain importance in raw NSE (high-flow weighted). Parameters below lose importance (more influential under variance-stabilised sqrt-NSE).

In [ ]:
comp['AbsΔ'] = comp['ΔShapley'].abs()
top = comp.sort_values('AbsΔ', ascending=False).head(6)

print('Top 6 parameters by Shapley shift:\n')
for _, r in top.iterrows():
    direction = '↑ raw (high-flow weighted)' if r['ΔShapley'] > 0 else '↓ raw (sqrt-stabilised)'
    print(f'  {r["Param"]:>6s}  {r["Sqrt"]:.6f}→{r["Raw"]:.6f}  ΔRk:{r["ΔRank"]:+d}  {direction}')

print(f'\nVariance ratio (raw/sqrt NSE): {Y_raw.var()/Y_sqrt.var():.1f}x')
print(f'Shapley sum — Sqrt: {sh_s.sum():.6f}  |  Raw: {sh_r.sum():.6f}')

### Results Summary

| | Sqrt-NSE | Raw NSE |
|---|:---:|:---:|
| Training mean | 0.784 | 0.692 |
| Training variance | 0.0021 | 0.0097 |
| Surrogate R² | 0.971 | 0.989 |
| Shapley sum | 0.00149 | 0.00553 |

Raw NSE has **4.7× more variance** than sqrt-NSE and a lower mean (0.69 vs 0.78), confirming that peak-flow errors dominate when the transform is removed.

### Top 5 by Shapley effect

| Rank | Sqrt-NSE | Raw NSE |
|:----:|----------|---------|
| 1 | **PFREE** (0.253) | **LZFSM** (0.203) |
| 2 | LZTWM (0.130) | LZFPM (0.166) |
| 3 | LZPK (0.111) | PFREE (0.127) |
| 4 | LZFSM (0.094) | LZPK (0.106) |
| 5 | ADIMP (0.090) | REXP (0.101) |

The top-5 lists share only LZPK and LZFSM. The correlation between sqrt and raw Shapley rankings is just **0.48** — the transform fundamentally reshapes the sensitivity structure.

### Key rank shifts

| Parameter | Sqrt rank | Raw rank | Shift | Interpretation |
|-----------|:---:|:---:|:---:|-----|
| **LZFPM** | 7 | **2** | **+5** | Primary free water is critical when high flows are weighted — it controls the volume of baseflow released after large storms. |
| **REXP** | 10 | **5** | **+5** | Percolation shape exponent controls the nonlinearity of vertical drainage. Under raw NSE, getting percolation right during wet periods matters far more than during dry periods. |
| **LZFSM** | 4 | **1** | **+3** | Supplementary free water becomes the single most important parameter. It's the faster-responding lower-zone store that fills during storms and drains via LZSK. |
| **LZTWM** | 2 | **7** | **−5** | Lower-zone tension water controls ET — a process whose errors (a few mm/day) are dwarfed by peak-flow errors (tens to hundreds of m³/s). |
| **ADIMP** | 5 | **10** | **−5** | Additional impervious area was expected to gain importance for raw NSE (it generates direct storm runoff). Instead it loses ground to the free-water parameters that control the *volume* of baseflow recession. |
| **PFREE** | 1 | **3** | **−2** | The percolation split remains important but cedes the top spot. Its influence is spread across both fast and slow pathways; raw NSE rewards parameters that specifically boost peak-recession accuracy. |

### Why the transform matters

NSE = 1 − Σ(Q_obs − Q_sim)² / Σ(Q_obs − Q̄_obs)²

With raw discharge, a 10 m³/s error at 500 m³/s contributes the same SSE as a 0.002 m³/s error at 0.1 m³/s. The largest ~250 days completely dominate the objective. The sqrt-transform compresses this: a 10× larger flow contributes only ~3.2× more to SSE.

The 0.48 Shapley correlation between sqrt and raw NSE means the choice of transform is not a cosmetic decision — it determines which parameters appear important. A model calibrated to raw NSE will prioritise free-water storage parameters (LZFSM, LZFPM) and percolation nonlinearity (REXP). A model calibrated to sqrt-NSE will weight PFREE and tension-water parameters (LZTWM) more heavily.

### Upper-zone parameters remain insensitive

UZFWM (rank 11→14), UZK (rank 13→13), and PCTIM (rank 12→11) are the least influential parameters regardless of transform. The upper zone fills and drains within days — its behaviour is dominated by forcing and lower-zone capacity. This result is consistent: calibrating these parameters adds little value for this catchment.

### Practical guidance

- **Use sqrt-NSE for calibration.** It's the standard in hydrological modelling for good reason — it prevents a handful of storm days from dominating the objective. The 0.48 correlation with raw NSE confirms that the transform is consequential.
- **If raw NSE must be used** (e.g., for flood-focused studies), prioritise LZFSM, LZFPM, LZPK, PFREE, and REXP. These five account for the majority of sensitivity.
- **REXP matters surprisingly much for raw NSE.** In sqrt-NSE it's rank 10 with negligible influence. In raw NSE it jumps to rank 5. Calibration studies using raw NSE should not fix REXP at a default value.
